# **Importar Librerias Necesarias**

In [0]:
%pip install flask pyngrok nltk tensorflow

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import re
import os
import warnings
warnings.filterwarnings("ignore")

import builtins
import threading

import nltk
import pickle
import numpy as np

from pyspark.ml.feature import StopWordsRemover
StopWordsRemover.loadDefaultStopWords("english")
from nltk.corpus import stopwords

from flask import Flask, request, redirect, url_for, render_template_string, session, jsonify

from tensorflow import keras
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

---


---
#<center> **Despliegue Modelo**</center>
---


---


En esta  fase, el enfoque se traslada desde la construcción y evaluación interna de los modelos hacia su despliegue en un entorno funcional, mediante una interfaz web que permite interactuar directamente con el sistema de clasificación desarrollado.

A través de esta implementación, será posible observar cómo el modelo procesa entradas textuales en tiempo real, aplicando automáticamente el flujo completo de preprocesamiento y generando predicciones comprensibles para el usuario final. Esto permite evaluar su comportamiento fuera del entorno controlado de entrenamiento y pruebas, evidenciando su capacidad de generalización ante datos nuevos y variados.

###**Cargar Modelo & Tokenizer**

En este apartado se cargan los recursos fundamentales necesarios para el funcionamiento del sistema de clasificación. Se importa el modelo CNN previamente entrenado, junto con el tokenizer utilizado durante el proceso de entrenamiento, garantizando coherencia en la representación numérica de los textos. Además, se inicializa el conjunto de stopwords en inglés y se define la longitud máxima de secuencia, parámetros esenciales para reproducir el mismo flujo de preprocesamiento aplicado originalmente a los datos.

In [0]:
# Cargar modelo CNN
model = keras.models.load_model("/Workspace/Users/jonatanr9605@gmail.com/ProyectoMaster/Modelos/cnn_text_model.keras")

# Cargar Tokenizer
with open("/Workspace/Users/jonatanr9605@gmail.com/ProyectoMaster/Tokenizer/tokenizer_cnn.pkl", "rb" ) as f:tokenizer = pickle.load(f)

# Stopwords
stopwords_en = set(StopWordsRemover.loadDefaultStopWords("english"))

# Longitud maxima de secuencias
max_len = 288

print("Modelo, Tokenizer y Stopwords cargados correctamente")

Modelo, Tokenizer y Stopwords cargados correctamente


###**Pipeline Pre-Procesamiento**

En este apartado se define una función que replica, a nivel local, el mismo flujo de preprocesamiento aplicado previamente en el entrenamiento del modelo. La función realiza la tokenización mediante expresiones regulares, elimina tokens vacíos, filtra las stopwords y descarta palabras de longitud menor a tres caracteres. Finalmente, reconstruye el texto limpio en formato de cadena, listo para ser transformado por el tokenizer en secuencias numéricas compatibles con el modelo.

In [0]:
# Función con el Pipeline de Pre-Procesamiento
def spark_like_pipeline(text: str) -> str:

    # RegexTokenizer
    tokens = re.split(r"[^a-zA-ZáéíóúñÁÉÍÓÚÑ]+", text.lower())

    # Quitar vacíos
    tokens = [t for t in tokens if t]

    # Stopwords
    tokens = [t for t in tokens if t not in stopwords_en]

    # Eliminar tokens < 3
    tokens = [t for t in tokens if len(t) >= 3]

    # Retornar texto
    return " ".join(tokens)

###**Diseño Interfaz Web**

En este apartado se define la estructura visual de la interfaz web mediante HTML y CSS embebido. Se diseña una página simple y moderna que permite al usuario ingresar un texto, enviarlo para su análisis y visualizar el resultado de la predicción. El estilo aplicado mejora la experiencia visual mediante colores, tipografía y distribución de los elementos, facilitando la interacción con el modelo de forma clara e intuitiva.

In [0]:
# Diseño HTML
HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Detector de Riesgo Suicida</title>
    <style>
        body{
            font-family: 'Segoe UI', sans-serif;
            background: linear-gradient(120deg,#1f2937,#111827);
            color:white;
            padding:40px;
        }
        textarea{
            width:700px;
            height:160px;
            padding:15px;
            border-radius:8px;
            border:none;
            font-size:16px;
        }
        button{
            display:block;
            margin-top:15px;
            padding:12px 30px;
            font-size:18px;
            border:none;
            border-radius:8px;
            background:#2563eb;
            color:white;
            cursor:pointer;
        }
        .result{
            margin-top:30px;
            font-size:26px;
            font-weight:bold;
        }
    </style>
</head>
<body>
    <h1>🧠 Detector Riesgo Suicida</h1>

    <form method="POST" autocomplete="off">
        <textarea id="text" name="text" placeholder="Escribe aquí el texto...">{{ text }}</textarea>
        <button type="submit">Analizar texto</button>
    </form>

    {% if result %}
        <div class="result">{{ result }}</div>
    {% endif %}

    <script>
    window.onload = function() {
        var txt = document.getElementById("text");
        txt.value = `{{ text | safe }}`;
    }
    </script>
</body>
</html>
"""

###**Desplegar Interfaz Web A Traves Flask**

En este apartado se implementa la lógica principal de la aplicación web utilizando Flask. Se define la ruta que recibe el texto ingresado por el usuario, aplica el pipeline de preprocesamiento, convierte el texto en secuencias numéricas mediante el tokenizer, realiza el padding según la longitud máxima definida y finalmente ejecuta la predicción con el modelo CNN. El resultado se transforma a porcentaje y se muestra en la interfaz. Además, la aplicación se ejecuta en un hilo independiente para permitir que Flask funcione en segundo plano dentro del entorno de trabajo.

In [0]:
# Aplicación con Flask
app = Flask(__name__)

@app.route("/", methods=["GET", "POST"])
def home():

    result = None
    text = ""

    if request.method == "POST":        
        text = request.form.get("text", "")

        # Pipeline
        clean_text = spark_like_pipeline(text)

        # Tokenizer
        seq = tokenizer.texts_to_sequences([clean_text])

        # Padding
        padded = pad_sequences(seq, maxlen=max_len, padding="post")

        # Predicción
        pred = model.predict(padded)[0][0]

        percent = pred * 100

        if pred > 0.5:
            result = f"⚠️ Riesgo suicida ({percent:.2f}%)"
        else:
            result = f"✅ No suicida ({percent:.2f}%)"

    return render_template_string(HTML, result=result, text=text)

# Función para ejecutar Flask en un hilo separado
def run():
    app.run(host="0.0.0.0", port=5000)

thread = threading.Thread(target=run)
thread.daemon = True
thread.start()

print("Flask ejecutándose en segundo plano...")

Flask ejecutándose en segundo plano...
 * Serving Flask app '__main__'
 * Debug mode: off


###**URL Interfaz Web**

En este apartado se obtiene dinámicamente la información del entorno de Databricks (workspace, cluster y host) para construir la URL que permite acceder a la aplicación Flask ejecutada en el puerto 5000. Esto facilita abrir la interfaz web directamente desde el navegador mediante el driver-proxy, sin necesidad de herramientas externas de túnel o configuraciones adicionales.

In [0]:
import dbruntime.databricks_repl_context as ctx

org_id = ctx.get_context().workspaceId
cluster_id = ctx.get_context().clusterId
host = ctx.get_context().browserHostName

print(f""" Abrir Aplicación Aquí: https://{host}/driver-proxy/o/{org_id}/{cluster_id}/5000/""")

 Abrir Aplicación Aquí: https://dbc-23af8dc1-252a.cloud.databricks.com/driver-proxy/o/258346112869825/0204-015105-trowpcos-v2n/5000/
